# Read-Well voice clone (free Colab GPU)
Clones **one consenting speaker's voice** with Chatterbox (MIT license) and reads every line Sunny says.

1. Runtime > Change runtime type > **T4 GPU**
2. Run the cells in order. When asked, upload `sample.wav` (10-30 s, one speaker, quiet room) and `lines.json`.
3. Download `voice-out.zip` at the end and follow `voice-clone/README.md` in the repo.

In [ ]:
!pip -q install chatterbox-tts
# Colab's torchvision needs a newer torch than Chatterbox installs, and we don't use it.
!pip -q uninstall -y torchvision
# Now Runtime > Restart session, then run the next cell.

In [ ]:
from google.colab import files
up = files.upload()   # choose sample.wav (or .mp3/.m4a) AND lines.json
sample = next(n for n in up if not n.endswith('.json'))
print('sample:', sample)

In [ ]:
import json, torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS
model = ChatterboxTTS.from_pretrained(device='cuda')
lines = json.load(open('lines.json'))
import os; os.makedirs('out', exist_ok=True)

def say(text, name):
    wav = model.generate(text, audio_prompt_path=sample, exaggeration=0.5, cfg_weight=0.5)
    ta.save(f'out/{name}.wav', wav, model.sr)

# Test one line first and listen before running them all.
say(lines[0]['text'], 'test')
from IPython.display import Audio; Audio('out/test.wav')

In [ ]:
from tqdm import tqdm
for l in tqdm(lines):
    torch.manual_seed(7)          # same seed each line = steadier voice
    say(l['text'], f"{l['id']:03d}")

In [ ]:
!cp lines.json out/ && zip -qr voice-out.zip out
from google.colab import files; files.download('voice-out.zip')